# 02 — Baseline model: class-weighted Logistic Regression
### CREDIT CARD FRAUD DETECTION SYSTEM

Trains the class-weighted logistic regression baseline on the same split used by `train_model.py` and compares it to the XGBoost model on the held-out test set. The baseline exists to make the advanced model's value measurable — if XGBoost does not beat it on PR-AUC and recall-at-threshold, we should say so.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('backend').resolve()))

import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit

from app.fraud_detector.config import DEFAULT_MODEL_CONFIG, TARGET
from app.fraud_detector.data.loading import load_dataset
from app.fraud_detector.features.build import FeaturePreprocessor, engineer_features
from app.fraud_detector.models.baseline import BaselineModel
from app.fraud_detector.models.xgboost_model import XGBFraudModel
from app.fraud_detector.evaluation.metrics import compute_metrics
from app.fraud_detector.utils.seeds import set_global_seed

set_global_seed(42)
cfg = DEFAULT_MODEL_CONFIG
df, _ = load_dataset()
X_raw, y = df.drop(columns=[TARGET]), df[TARGET].to_numpy()

idx_train, idx_test = next(StratifiedShuffleSplit(n_splits=1, test_size=cfg.test_size, random_state=42).split(X_raw, y))
train_df, test_df = df.iloc[idx_train], df.iloc[idx_test]
y_train, y_test = y[idx_train], y[idx_test]
print(f"train={len(train_df)} test={len(test_df)} | fraud rate test={y_test.mean():.5f}")

In [ ]:
# Preprocessing fitted on TRAIN only — identical to the training pipeline.
pp = FeaturePreprocessor().fit(engineer_features(train_df))
X_train = pp.transform(engineer_features(train_df))
X_test = pp.transform(engineer_features(test_df))

baseline = BaselineModel(cfg).fit(X_train, y_train)
xgb = XGBFraudModel(cfg).fit(X_train, y_train)

b = compute_metrics(y_test, baseline.predict_proba(X_test), 0.5)
x = compute_metrics(y_test, xgb.predict_proba(X_test), 0.5)
print('Baseline LR @0.5  -> precision %.4f recall %.4f f1 %.4f PR-AUC %.4f ROC-AUC %.4f' % (b['precision'], b['recall'], b['f1'], b['pr_auc'], b['roc_auc']))
print('XGBoost      @0.5  -> precision %.4f recall %.4f f1 %.4f PR-AUC %.4f ROC-AUC %.4f' % (x['precision'], x['recall'], x['f1'], x['pr_auc'], x['roc_auc']))

In [ ]:
# At the threshold the pipeline actually selects (validation-optimized):
import json
from app.fraud_detector.utils.artifacts import active_version, read_json, version_dir

version = active_version()
if version:
    thresholds = read_json(version_dir(version) / 'thresholds.json')
    t = thresholds['best_threshold']
    b2 = compute_metrics(y_test, baseline.predict_proba(X_test), t)
    x2 = compute_metrics(y_test, xgb.predict_proba(X_test), t)
    print(f'--- at selected threshold {t:.4f} ---')
    print('Baseline LR -> precision %.4f recall %.4f f1 %.4f' % (b2['precision'], b2['recall'], b2['f1']))
    print('XGBoost     -> precision %.4f recall %.4f f1 %.4f' % (x2['precision'], x2['recall'], x2['f1']))
    print('\nInterpretation: the XGBoost model is expected to dominate on precision at comparable recall;')
    print('if it does not, the training report says so honestly.')